# Intro: A Minimal Local Eval Harness

This notebook shows the basic learning loop: load a small dataset, run a deterministic mock model, score the outputs, aggregate the results, and write a report.

In [ ]:
from pathlib import Path
import json

from eval_harness.aggregation import summarize_records
from eval_harness.dataset import load_jsonl_cases
from eval_harness.metrics import exact_match, normalized_exact_match, pass_fail_score
from eval_harness.reporting import write_json_report
from eval_harness.runner import run_cases

root = Path.cwd()
cases = load_jsonl_cases(root / "tests" / "evals" / "datasets" / "classification_cases.jsonl")
mock_outputs = json.loads((root / "tests" / "evals" / "mocks" / "mock_model_outputs.json").read_text())

def predict(case):
    return mock_outputs[case.case_id]

def score(case, actual):
    exact = pass_fail_score(exact_match(actual, case.expected))
    normalized = pass_fail_score(normalized_exact_match(actual, case.expected))
    passed = bool(normalized if case.metadata.get("match_type") == "normalized" else exact)
    return {"exact_match": exact, "normalized_exact_match": normalized}, passed

records = run_cases(cases, predict, scorer=score)
summary = summarize_records(records)
write_json_report(root / "outputs" / "notebook_intro_report.json", records, summary)
summary.model_dump()
